# 02 — Reading & Writing Data

**What you will learn:**
- Reading CSV, JSON, Parquet, and Delta formats
- Common reader options (header, inferSchema, delimiter, schema)
- Writing DataFrames with different save modes
- Writing to Delta Lake (managed and external tables)
- Checking partition files written to storage
- Using Unity Catalog Volumes as a practice storage layer

## Prerequisites — Unity Catalog Volume Setup

> **Why Volumes?**
> DBFS `/tmp/` paths and raw `abfss://` paths require either legacy DBFS access (disabled on new Unity
> Catalog clusters) or a fully configured External Location + Access Connector.  
> **Unity Catalog Volumes** are the modern, zero-extra-setup way to store raw files in Databricks — 
> they are backed by cloud storage that Databricks already manages for you.

### One-time setup: create a Volume

Do this **once** before running any cell in this notebook:

1. In the Databricks left sidebar, click **Catalog** (the grid icon).
2. Expand **main** catalog → **default** schema (these exist in every new Unity Catalog workspace).
3. Click **Create Volume**.
4. Set **Volume name** to `practice_vol` and click **Create**.

The volume path you will use everywhere in this notebook is:

```
/Volumes/main/default/practice_vol/
```

If your workspace uses a different default catalog name (check Catalog Explorer — it is usually `main`),
change the `CATALOG` variable in the next cell.

> **Uploading files manually (optional)**  
> You can also upload CSV/JSON files directly through the UI:  
> Catalog → main → default → practice_vol → **Upload to this volume** button.  
> This notebook creates the sample files programmatically so you do not need to upload anything.

In [ ]:
# ── Change these two values if your workspace uses different names ──
CATALOG = "main"
SCHEMA  = "default"
VOLUME  = "practice_vol"

BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
print(f"All files in this notebook go to: {BASE}")

## Setup — Create Sample Files in the Volume

In [ ]:
import json

# Write a sample CSV to the volume
csv_data = """emp_id,name,department,salary,hire_date
1,Alice,Engineering,95000,2022-01-15
2,Bob,Marketing,72000,2021-06-01
3,Charlie,Engineering,105000,2020-03-20
4,Diana,HR,68000,2023-09-10
5,Eve,Marketing,78000,2022-11-05
6,Frank,Engineering,88000,2021-04-12
7,Grace,HR,71000,2022-07-30
"""

dbutils.fs.put(f"{BASE}/employees.csv", csv_data, overwrite=True)

# Write sample JSON (newline-delimited JSON = JSON Lines)
json_data = "\n".join([
    json.dumps({"order_id": 1001, "customer": "Alice", "amount": 250.50, "status": "COMPLETED"}),
    json.dumps({"order_id": 1002, "customer": "Bob",   "amount": 89.99,  "status": "PENDING"}),
    json.dumps({"order_id": 1003, "customer": "Alice", "amount": 430.00, "status": "COMPLETED"}),
    json.dumps({"order_id": 1004, "customer": "Diana", "amount": 120.75, "status": "CANCELLED"}),
])
dbutils.fs.put(f"{BASE}/orders.json", json_data, overwrite=True)

# Verify the files are there
display(dbutils.fs.ls(BASE))
print("Sample files created in volume.")

## 1. Reading CSV

In [ ]:
# --- Basic CSV read with options ---
df_csv = (
    spark.read
    .option("header", "true")          # first row = column names
    .option("inferSchema", "true")     # auto-detect data types (slow on large files)
    .option("delimiter", ",")          # default is comma
    .option("nullValue", "NA")         # treat "NA" string as null
    .option("emptyValue", "")          # treat empty string as empty (not null)
    .csv(f"{BASE}/employees.csv")
)

df_csv.printSchema()
df_csv.show()

In [ ]:
# --- Better: define schema explicitly (faster, no scan) ---
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

emp_schema = StructType([
    StructField("emp_id",     IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("hire_date",  StringType(),  True),
])

df_emp = (
    spark.read
    .option("header", "true")
    .schema(emp_schema)
    .csv(f"{BASE}/employees.csv")
)

df_emp.printSchema()
df_emp.show()

## 2. Reading JSON (Newline-Delimited)

In [ ]:
df_json = (
    spark.read
    .option("multiLine", "false")   # false = each line is one JSON object (JSON Lines)
    .json(f"{BASE}/orders.json")
)

df_json.printSchema()
df_json.show()

## 3. Writing Data — Save Modes

| Mode | Behavior |
|---|---|
| `overwrite` | Delete existing data and write new |
| `append`    | Add new data alongside existing data |
| `ignore`    | Do nothing if path already exists |
| `error` (default) | Throw error if path already exists |

In [ ]:
# Write to Parquet (columnar, compressed — preferred format for data lakes)
(
    df_emp
    .write
    .mode("overwrite")
    .parquet(f"{BASE}/employees_parquet/")
)

print("Written to Parquet.")

In [ ]:
# Write to CSV
(
    df_emp
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(f"{BASE}/employees_out_csv/")
)
print("Written to CSV.")

In [ ]:
# Write to JSON
(
    df_emp
    .write
    .mode("overwrite")
    .json(f"{BASE}/employees_out_json/")
)
print("Written to JSON.")

## 4. Reading Parquet

In [ ]:
df_parquet = spark.read.parquet(f"{BASE}/employees_parquet/")
df_parquet.printSchema()
df_parquet.show()

## 5. Writing & Reading Delta Lake

In [ ]:
# Write as Delta
(
    df_emp
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{BASE}/employees_delta/")
)

print("Written as Delta.")

In [ ]:
# Read Delta
df_delta = spark.read.format("delta").load(f"{BASE}/employees_delta/")
df_delta.show()

## 6. Partitioned Writes

Partitioning organizes files into sub-folders by column values.  
Queries that filter on the partition column skip irrelevant files (partition pruning).

In [ ]:
(
    df_emp
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("department")          # one sub-folder per department
    .save(f"{BASE}/employees_partitioned/")
)

# List the partition folders
display(dbutils.fs.ls(f"{BASE}/employees_partitioned/"))

In [ ]:
# Read with partition filter — Spark reads only the Engineering folder
df_eng = (
    spark.read
    .format("delta")
    .load(f"{BASE}/employees_partitioned/")
    .filter("department = 'Engineering'")
)
df_eng.show()

## 7. Creating Delta Tables in Unity Catalog

Writing to a path creates a Delta file.  
Registering it as a table lets you query it with SQL using `catalog.schema.table` syntax.

In [ ]:
# Managed table — Databricks controls the storage location
# Unity Catalog stores it in the metastore-managed storage automatically
df_emp.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.employees")

# Read it back via SQL
spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.employees").show()

In [ ]:
# External table backed by the volume path
# Volume path is the storage location — dropping the table does NOT delete the files
(
    df_emp.write
    .format("delta")
    .mode("overwrite")
    .option("path", f"{BASE}/ext_employees/")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.ext_employees")
)

spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.ext_employees").show()

## 8. Reading Multiple Files at Once

In [ ]:
# Read all CSV part-files written by Spark in a folder using wildcard
# spark.read.csv(f"{BASE}/employees_out_csv/*.csv")

# Add source file name as a column (useful for auditing)
from pyspark.sql.functions import input_file_name

df_with_source = (
    spark.read
    .option("header", "true")
    .csv(f"{BASE}/employees.csv")
    .withColumn("source_file", input_file_name())
)
df_with_source.show(truncate=False)

## 9. Reading ADLS Gen2 Directly — `abfss://` Paths

> **Prerequisites (requires additional setup):**  
> Reading `abfss://` paths directly from a notebook requires **one of the following**:
> - Unity Catalog **External Location** + **Storage Credential** (Access Connector with managed identity) — recommended for production
> - Spark session-level storage account key — acceptable for exploration only
>
> Neither works out of the box without that setup.  
> Use the Volume path (`/Volumes/...`) above for all practice exercises.  
> The cells below are **reference templates** — replace placeholders before running.

### Method A — Unity Catalog External Location (production)

If your workspace admin has configured an External Location pointing to the storage account,
no credentials are needed in the notebook at all:

In [ ]:
# Replace <your-storage-account> with your actual Azure storage account name
# This only works after an admin has created an External Location in Unity Catalog

# display(dbutils.fs.ls("abfss://bronze@<your-storage-account>.dfs.core.windows.net/"))

# df_adls = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv("abfss://bronze@<your-storage-account>.dfs.core.windows.net/employees.csv")
# )
# df_adls.show()

print("Uncomment the lines above after External Location is configured.")

### Method B — Storage Account Key (exploration only)

Set the account key once per session in `spark.conf`, then use the same `abfss://` path.  
**Never commit account keys to version control.**

**Where to find the key:**  
Azure Portal → Storage Account → Security + Networking → **Access keys** → copy key1

In [ ]:
# Replace both placeholders before running
# STORAGE_ACCOUNT = "<your-storage-account>"
# ACCOUNT_KEY      = "<your-account-key>"

# spark.conf.set(
#     f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
#     ACCOUNT_KEY
# )

# df_key_csv = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv(f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/employees.csv")
# )
# df_key_csv.show()

print("Uncomment and fill in the placeholders to use storage account key auth.")

## Storage Path Options — When to Use Which

| Path Type | Example | Setup Required | Best For |
|---|---|---|---|
| **Unity Catalog Volume** | `/Volumes/main/default/vol/file.csv` | Create a Volume (UI, 1 min) | All practice, dev work, file staging |
| **abfss:// + External Location** | `abfss://container@acct.dfs.core.windows.net/` | Admin configures Storage Credential + External Location | Production pipelines, org-wide data lake |
| **abfss:// + Account Key** | same path, key set via `spark.conf` | Copy key from Azure Portal | Quick personal exploration only |
| **DBFS /tmp/** | `/tmp/practice/file.csv` | None | Legacy only — disabled on Unity Catalog clusters |

> **Rule of thumb for interviews:** say "I'd use Unity Catalog Volumes for staging raw files, 
> and External Locations with managed identity for production ADLS access — no keys in notebooks."

## Key Takeaways

| Operation | Code Pattern |
|---|---|
| Read CSV | `spark.read.option("header","true").schema(s).csv(path)` |
| Read JSON | `spark.read.json(path)` |
| Read Parquet | `spark.read.parquet(path)` |
| Read Delta | `spark.read.format("delta").load(path)` |
| Write overwrite | `.write.mode("overwrite").parquet(path)` |
| Write partitioned | `.write.partitionBy("col").save(path)` |
| Managed table (UC) | `.write.saveAsTable("catalog.schema.table")` |
| External table (UC) | `.write.option("path", volume_path).saveAsTable("catalog.schema.table")` |
| Volume path format | `/Volumes/<catalog>/<schema>/<volume>/<file>` |
| Always prefer explicit schema | Avoids full-scan inference, enforces types |